GMM Based Generation

Reproduces Section 5.2 of the paper. Fits a GMM to channel-wise mean RGB colors of the CelebA training set, samples flat-color images form it, and uses Algorithm 2 to "deblur" them into generated faces. Compares with and without symmetry-breaking noise(std= 0.002) per paper Table 5 (CelebA: FID 97.00 -> 49.45 with noise).

In [ ]:
from google.colab import drive
import matplotlib.pyplot as plt

drive.mount("/content/drive")
repo= "/content/drive/MyDrive/cs4782-final-project"

%cd {repo}

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
from data.datasets import get_loader
from src.model import build_unet
from src.degradation import Degradation
from src.generation import fit_gmm, generate_image
from src.evaluate import compute_fid, to_fid

device= "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)

In [ ]:
T= 300
IMAGE_SHAPE= (3, 64, 64)
# 10k would give paper comparable FID but takes around 5 times longer
N_SAMPLES= 2000 

CKPT= "./checkpoints/celeba_gen.pt"



In [ ]:
model = build_unet("celeba").to(device)
ckpt= torch.load(CKPT, map_location= device)

model.load_state_dict(ckpt["ema"])
model.eval()

degradation= Degradation(
    kernel_size= 27, num_timesteps= T, kernel_std= 0.01,
    blur_routine= "Exponential", image_channels=3, 
).to(device)

train_loader= get_loader("celeba", "train", batch_size= 64, num_workers=2, shuffle= False)
test_loader= get_loader("celeba", "test", batch_size= 64, num_workers=2, shuffle= False)

gmm= fit_gmm(train_loader, n_components= 1) 
print("GMM means: ", gmm.means_)



In [ ]:
import torch

def gen_batched(n_total, batch_size, add_noise):
    out= []
    for i in range(0, n_total, batch_size):
        n = min(batch_size, n_total - i)

        chunk= generate_image(model, degradation, gmm, t= T, image_shape= IMAGE_SHAPE, n_samples=n, add_noise= add_noise, device= device)
        # move off gpu to free memory between batches
        out.append(chunk.cpu())

    return torch.cat(out)

generate_no_noise= gen_batched(N_SAMPLES, batch_size= 256, add_noise= False)
generate_noise= gen_batched(N_SAMPLES, batch_size= 256, add_noise= True)
print("generate_no_noise", generate_no_noise.shape)
print("generate_noise", generate_noise.shape)

In [ ]:
# compute fid against real test images

reals= []
for images, _ in test_loader: 
    reals.append(images.to(device))
    if sum(r.shape[0] for r in reals) >= N_SAMPLES:
        break
reals = torch.cat(reals)[:N_SAMPLES]

reals= reals.cpu()

real_u8 = to_fid(reals)
no_noise_u8= to_fid(generate_no_noise)
noise_u8= to_fid(generate_noise)

fid_no_noise = compute_fid(real_u8, no_noise_u8)
fid_noise = compute_fid(real_u8, noise_u8)

print(f"FID (no noise): {fid_no_noise:.2f}")
print(f"FID (with 0.002 noise): {fid_noise:.2f}")
print(f"Improvement (from without to with noise): {fid_no_noise - fid_noise:.2f}")





In [ ]:
# FIgure

vis_no_noise = generate_image(model, degradation, gmm, t= T, image_shape= IMAGE_SHAPE, n_samples= 8, add_noise= False, device= device)
vis_noise = generate_image(model, degradation, gmm, t= T, image_shape= IMAGE_SHAPE, n_samples= 8, add_noise= True, device= device)

# resample colors so we can show what was fed in

colors, _ = gmm.sample(8)
colors= torch.from_numpy(colors).float().clamp(0, 1)

flat_colors = colors.view(8, 3, 1, 1).expand(8, 3, 64, 64)

fig, axes= plt.subplots(3, 8, figsize= (16, 6))
for i in range(8):
    axes[0, i].imshow(flat_colors[i].permute(1, 2, 0)); axes[0, i].axis("off")
    axes[1, i].imshow(vis_no_noise[i].clamp(0, 1).cpu().permute(1, 2, 0)); axes[1, i].axis("off")
    axes[2, i].imshow(vis_noise[i].clamp(0, 1).cpu().permute(1, 2, 0)); axes[2, i].axis("off")

#row labels
for ax, label in zip(axes[:, 0], ["flat color", "no noise", "+0.002 noise"]):
    ax.set_ylabel(label, fontsize= 12)
    ax.axis("on"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig("./results/generation_samples.png", dpi= 150, bbox_inches= "tight")
plt.show()

In [ ]:
import json, os
os.makedirs("./results", exist_ok = True)

with open("./results/generation_fid.json", "w") as f: 
    json.dump({
        "fid_no_noise": fid_no_noise,
        "fid_with_noise": fid_noise,
        "n_samples": N_SAMPLES,
        "gmm_n_components": 1, 
        "T":T,
    }, f, indent= 2)

print("saved ot ./results/")